# Real data, step by step

This notebook fits one region of the ex vivo macaque data (slice 42), and explains each step.

**The idea.** In every voxel we measure the signal at several b-values and echo times (TE). We assume each voxel is a mix of a few tissue *compartments*. Each compartment has its own spectrum over a grid of diffusivities $D$ and $T_2$ values:

$$Y \approx M_0 \cdot A \, C \, W$$

| Symbol | Meaning | Shape |
|---|---|---|
| $Y$ | measured signal, one column per voxel | (measurements, voxels) |
| $A$ | signal model: what each grid point $(D, T_2)$ gives at each $(b, TE)$ | (measurements, grid points) |
| $C$ | spectra of the $K$ compartments (shared by all voxels) | (grid points, $K$) |
| $W$ | how much of each compartment is in each voxel (sums to 1) | ($K$, voxels) |
| $M_0$ | overall signal scale per voxel | (voxels,) |

We know $Y$ and $A$, and we estimate $C$, $W$ and $M_0$.

## 1. Setup

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib

root = Path.cwd().resolve(); root = root if (root / 'src').exists() else root.parent
sys.path.insert(0, str(root))

from src.create_grid import apply_SVD_to_A
from src.optimiser_alpha_gamma import project_C_to_C_small, initialize_W_from_C, run_constrained_optimisation
from src.signal_models import DiffusionT2Model
from src.plotting import show_canonical_spectra, plot_voxel_spectra_and_weights, plot_voxel_fit, plot_loss_curve

## 2. Load the data

The signal matrix $Y$ was built beforehand (the first code cell of `maria-tests-real-data_for_poster_2.ipynb`) and saved to disk. That step:
1. reads slice 42 from each of the 6 runs (one TE per run),
2. removes the **noise floor** from each volume: $\sqrt{\max(M^2 - \overline{\text{background}^2},\, 0)}$,
3. averages the volumes that share a b-value (**powder average**).

So here we only load the result: 6 TEs × 8 b-values = **48 measurements** for every brain voxel in the slice.

In [ ]:
DATA_DIR = "/Users/mb4725/Library/CloudStorage/OneDrive-ImperialCollegeLondon/PhD/Code/Dataset_macaque"
OUT = f"{DATA_DIR}/preprocessed"
SLICE_IDX = 42

Y_all = np.load(f"{OUT}/Y_slice42_brain_nc.npy")             # (48, all brain voxels)
acq_table = np.load(f"{OUT}/acq_table_slice42_brain.npy")    # one row per measurement: b [s/mm²], TE [ms]
voxel_xy_all = np.load(f"{OUT}/voxel_xy_slice42_brain.npy")  # (x, y) position of each column of Y_all

b_list = acq_table[:, 0] / 1000.0    # s/mm² -> ms/µm², the same units as D
TE_list = acq_table[:, 1]

print("Y_all:", Y_all.shape)
print("b values (ms/µm²):", np.unique(b_list))
print("TE values (ms):  ", np.unique(TE_list))

For pictures we also load the white-matter (WM) mask and a b = 0 image of the slice.

In [ ]:
mask_slice = nib.load(f"{OUT}/WM_mask_dwispace.nii.gz").get_fdata()[:, :, SLICE_IDX] > 0

run1 = nib.load(f"{DATA_DIR}/sub-M1_acq-MultiDim_run-1_dwi.nii.gz")
bvals_run1 = np.loadtxt(f"{DATA_DIR}/sub-M1_acq-MultiDim_run-1_dwi.bval").ravel()
b0_slice = np.asarray(run1.dataobj[:, :, SLICE_IDX, :], dtype=np.float32)[..., bvals_run1 <= 50].mean(-1)

plt.imshow(b0_slice.T, cmap="gray", origin="lower")
plt.imshow(np.ma.masked_where(~mask_slice.T, mask_slice.T), cmap="autumn", alpha=0.3, origin="lower")
plt.title(f"Slice {SLICE_IDX}: b = 0 image, WM mask in orange"); plt.axis("off"); plt.show()

## 3. Choose a region

We fit a small patch of voxels at a time. Here it's the **40 voxels closest to a point** on the left WM/GM boundary, so the patch contains both white and grey matter.

In [ ]:
centre = (33, 51)     # (x, y) in voxels: left WM/GM boundary
n_voxels = 40

near = np.argsort(np.hypot(voxel_xy_all[:, 0] - centre[0], voxel_xy_all[:, 1] - centre[1]))[:n_voxels]
voxel_xy = voxel_xy_all[near]
is_wm = mask_slice[voxel_xy[:, 0], voxel_xy[:, 1]]
print(f"{n_voxels} voxels: {is_wm.sum()} in WM, {(~is_wm).sum()} outside WM")

plt.imshow(b0_slice.T, cmap="gray", origin="lower")
plt.imshow(np.ma.masked_where(~mask_slice.T, mask_slice.T), cmap="autumn", alpha=0.3, origin="lower")
plt.scatter(voxel_xy[:, 0], voxel_xy[:, 1], s=6, c="cyan")
plt.title("Selected voxels (cyan)"); plt.axis("off"); plt.show()

## 4. The signal model

The model says what signal one grid point $(D, T_2)$ gives at one measurement $(b, TE)$:

$$S = e^{-b D} \cdot e^{-TE / T_2}$$

`DiffusionT2Model` holds the grid (40 × 40 points) and builds $A$, with one column per grid point. **To try a different model, this is the only line you'd change.**

We also sort the measurements by b, then TE. That's only for tidiness; $Y$ is sorted the same way, so they still line up.

In [ ]:
model = DiffusionT2Model(Dmin=0.01, Dmax=1.0, nD=40, T2min=15.0, T2max=50.0, nT2=40)

A = model.A({"b": b_list, "TE": TE_list})        # (48, 1600)

order = np.lexsort((TE_list, b_list))             # sort by b, then TE
A, b_list, TE_list = A[order], b_list[order], TE_list[order]
Y = Y_all[:, near][order]                          # (48, 40): this region, same row order as A

print("A:", A.shape, "  Y:", Y.shape)

## 5. Compress A with an SVD

$A$ has 1,600 columns, but its columns are very similar to each other, so most of that information is redundant. The SVD keeps only the **R = 10 most important directions**: $A \approx U \,\text{diag}(s)\, V^T$. The fit works in this small space, which is much faster.

In [ ]:
R = 10
U, s, Vmat = apply_SVD_to_A(A, R)

print("singular values:", np.round(s / s[0], 4))

## 6. Starting values

The fit needs a starting guess for everything.

- **C (spectra):** fit every voxel on its own with NNLS, average those spectra, then split the average into $K$ regions with k-means. Each region becomes one starting compartment.
- **M0:** fit $M_0 e^{-TE/T_2}$ to the b ≈ 0 data of each voxel.
- **W (weights):** the best weights given those C and M0.
- **σ² (noise level):** from how badly that starting guess fits the data.

In [ ]:
K = 2    # number of compartments

C0, F_tissue, *_ = model.initial_C(Y, A, K, min_fraction=0.01, return_diagnostics=True)
M0_init = model.initial_M0(Y, {"b": b_list, "TE": TE_list})

Csmall0 = project_C_to_C_small(C0, s, Vmat)                    # C in the compressed SVD space
W0 = initialize_W_from_C(Y, U, Csmall0, M0_init, bound_eps=1e-3)

sigma2_init = np.mean((Y - (U @ Csmall0 @ W0) * M0_init) ** 2)

show_canonical_spectra([("Initial", C0)], model)

## 7. Fit

The optimiser takes turns updating each unknown while keeping the others fixed, for `n_iter` rounds. The main settings:

| Setting | What it does |
|---|---|
| `alpha` (learned) | Dirichlet prior on W: small = voxels dominated by one compartment |
| `m0_prior`, `m0_prior_sigma` | keeps M0 close to the starting value (within ~10%) |
| `c_smoothness` | penalises jagged spectra, so C is smooth over the grid |
| `alpha_update_start` | wait a few rounds before learning alpha, so C and W settle first |

This takes a few minutes.

In [ ]:
W_est, M0_est, C_est, alpha_est, sigma2_est, lam_est, losses, fit_errs = run_constrained_optimisation(
    Y, U, s, Vmat, C0.copy(), M0_init, K=K, n_iter=1000,
    sigma2=sigma2_init, lam=1e-2, alpha=1.0, W=W0.copy(),
    update_W_flag=True, update_C_flag=True, update_M0_flag=True, update_sigma2_flag=True,
    update_alpha_flag=True, alpha_update_start=5, weight_entropy=0.0,
    m0_prior=M0_init, m0_prior_sigma=0.1 * np.median(M0_init),
    c_smoothness=1000.0, c_sparsity=0.0, c_diversity=0.0, c_maxiter=100,
    nD=model.shape[0], nT2=model.shape[1],
    verbose_every=100,
)

plot_loss_curve(losses)

## 8. Results

### Compartment spectra
The two spectra shared by all voxels: initial guess vs. after fitting.

In [ ]:
show_canonical_spectra([("Initial", C0), ("Estimated", C_est)], model)

### Spectra and weights in a few voxels
Each voxel's spectrum is its own mix of the compartments: $F = C\,W$.

In [ ]:
F_est = C_est @ W_est
plot_voxel_spectra_and_weights(model, F_est, W_est, voxels=[5, 15, 25], fs=14)

### Weights on the brain
If the compartments mean something physical, the weights should follow the WM mask.

In [ ]:
zoom = 15
fig, axes = plt.subplots(1, K, figsize=(6 * K, 5), constrained_layout=True)
for k, ax in enumerate(axes):
    weight_map = np.full(mask_slice.shape, np.nan)
    weight_map[voxel_xy[:, 0], voxel_xy[:, 1]] = W_est[k]
    ax.imshow(b0_slice.T, cmap="gray", origin="lower")
    ax.imshow(np.ma.masked_where(~mask_slice.T, mask_slice.T), cmap="autumn", alpha=0.3, origin="lower")
    im = ax.imshow(np.ma.masked_invalid(weight_map.T), cmap="Blues", vmin=0, vmax=1,
                   origin="lower", interpolation="nearest")
    ax.set_xlim(voxel_xy[:, 0].min() - zoom, voxel_xy[:, 0].max() + zoom)
    ax.set_ylim(voxel_xy[:, 1].min() - zoom, voxel_xy[:, 1].max() + zoom)
    ax.set_title(f"Weight of component {k + 1}"); ax.axis("off")
fig.colorbar(im, ax=axes, fraction=0.046)
plt.show()

for k in range(K):
    print(f"component {k + 1}: mean weight in WM {W_est[k, is_wm].mean():.2f}, "
          f"outside WM {W_est[k, ~is_wm].mean():.2f}")

### How well does the model fit the data?
Dots = measured, lines = model prediction, one colour per TE.

In [ ]:
Y_pred = M0_est * (A @ (C_est @ W_est))

wm_voxel = np.where(is_wm)[0][0]
gm_voxel = np.where(~is_wm)[0][0]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_voxel_fit(Y, Y_pred, wm_voxel, b_list, TE_list, "WM voxel", ax=axes[0])
plot_voxel_fit(Y, Y_pred, gm_voxel, b_list, TE_list, "Outside-WM voxel", ax=axes[1])
plt.tight_layout(); plt.show()

print(f"Relative residual over the whole region: {np.linalg.norm(Y - Y_pred) / np.linalg.norm(Y):.3f}")

## Summary


**To try another region:** change `centre` in step 3.
**To try another number of compartments:** change `K` in step 6.
**To try another signal model:** write a new class in `src/signal_models.py` and change the `model = ...` line in step 4.